# 12-1 transformers 라이브러리로 LLM 다루기

본 노트북은 12-1절 본문 예제 [코드 12-1]~[코드 12-7]을 모아 위에서 아래로
실행한다. 허깅페이스 transformers 라이브러리로 한국어 LLM(Bllossom-3B)을
불러와, 토크나이저로 인코딩하고, 대화틀을 구성해 답변을 생성하는 전 과정을 다룬다.

**이 장의 코드 컨벤션 알림** - 12장은 허깅페이스 라이브러리를 중심 주제로
다루므로, `MODEL_NAME` 등 상수 표기는 공통 컨벤션을 따르지만 `model`,
`tokenizer`, `generator` 같은 객체명은 HF 통용 표기를 그대로 사용한다.

다루는 내용
- [코드 12-2] from_pretrained()로 모델과 토크나이저 불러오기
- [코드 12-1] pipeline()으로 LLM에 질문 던지기
- [코드 12-3] 한국어 토크나이저의 토큰화 결과
- [코드 12-4] 토크나이저 호출 방식과 반환값
- [코드 12-5] 대화틀 함수
- [코드 12-6] 대화 메시지를 모델 입력 텐서로 변환
- [코드 12-7] LLM에 대화 메시지를 입력해 답변 생성

> 사용 모델: Bllossom/llama-3.2-Korean-Bllossom-3B (Llama 3.2 커뮤니티
> 라이선스), gogamza/kobart-base-v2 (MIT 라이선스).


In [1]:
# 참고 - 라이브러리 설치 (이미 설치된 경우 건너뛰어도 좋다)
# !pip install -q transformers


In [1]:
# 참고 - 공통 라이브러리 경로 설정
import sys
sys.path.append('../../')

from code_reference import common

device = common.get_device()


CUDA를 사용합니다.


In [ ]:
# 참고 - 라이브러리 import 와 시드 고정
SEED = 42
common.set_seed(SEED)
device = common.get_device()

# # 참고 - 라이브러리 import 와 시드 고정
# import random
# import warnings

# import numpy as np
# import torch
# from transformers import (
#     AutoModelForCausalLM, AutoTokenizer, pipeline,
# )

# # 다운로드/로딩 시 경고 메시지를 줄인다 (정상 동작에는 영향 없음).
# warnings.filterwarnings('ignore')

# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)


## [코드 12-2] from_pretrained()로 모델과 토크나이저 불러오기

`pipeline()`보다 생성 과정을 직접 제어하려면 모델과 토크나이저를 따로 불러온다.
이후 셀에서 재사용하기 위해 먼저 불러온다. `torch_dtype=torch.float16`으로
지정해 FP32 대비 메모리를 절반으로 줄인다.


In [ ]:
###############################################################################
# 코드 12-2 - from_pretrained()로 모델과 토크나이저 불러오기
###############################################################################
# 허깅페이스 허브의 모델 식별자를 그대로 지정
MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# torch_dtype=torch.float16 으로 FP16 로딩 (생략 시 FP32라 메모리 두 배)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
).to(device)
model.eval()

print(f'모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'토크나이저 어휘 크기: {tokenizer.vocab_size:,}')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

모델 파라미터 수: 3,212,749,824
토크나이저 어휘 크기: 128,000


## [코드 12-1] pipeline()으로 LLM에 질문 던지기

`pipeline()`에 작업 이름과 모델만 넘기면 토크나이저 준비부터 디코딩까지 한 번에
처리한다. 본문 [코드 12-1]은 `pipeline('text-generation', model='Bllossom/...')`
처럼 모델 식별자를 직접 넘기지만, 노트북에서는 같은 모델을 두 번 적재하지 않도록
앞에서 FP16으로 불러둔 `model`, `tokenizer` 객체를 재사용한다(12GB GPU에서
모델을 두 번 올리면 메모리가 부족할 수 있다).


In [6]:
# 코드 12-1 - pipeline()으로 LLM에 질문 던지기
# 이미 불러둔 model, tokenizer 객체를 그대로 재사용한다.
generator = pipeline(
    'text-generation', model=model, tokenizer=tokenizer,
)
# 파이프라인을 실행하면 리스트 형태로 결과가 반환된다.
result = generator(
    '오픈 소스 모델의 주요 라이선스는 어떤 것이 있어?',
    max_new_tokens=128,
    do_sample=False,
)
print(result[0]['generated_text'])


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_wi

오픈 소스 모델의 주요 라이선스는 어떤 것이 있어? 

오픈 소스 모델의 주요 라이선스는 다음과 같습니다:

1. **MIT 라이선스 (MIT License)**: 이 라이선스는 소프트웨어를 사용하고 배포할 수 있으며, 소프트웨어의 원본 코드를 변경할 수 있습니다. 또한, 소프트웨어를 배포하고 사용하는 데 대한 모든 권리를 소유하고 있습니다.
2. **BSD 라이선스 (BSD License)**: 이 라이선스는 소프트웨어를 사용하고 배포할 수 있으며, 소프트웨어의 원본 코드를 변경할 수 있습니다. 또한, 소프트웨어를 배포하고


## [코드 12-3] 한국어 토크나이저의 토큰화 결과

토크나이저의 `tokenize()` 메서드는 고유 번호가 아닌 토큰 문자열의 리스트를
반환한다. 한국어는 교착어라 한 단어가 둘 이상의 토큰으로 나뉜다. 특수문자 ▁은
단어 경계(공백)를 나타낸다.


In [4]:
# 코드 12-3 - 한국어 토크나이저의 토큰화 결과
# gogamza/kobart-base-v2 모델: 한국어 BART 모델 (12-2절에서 자세히 소개)
kobart_tokenizer = AutoTokenizer.from_pretrained('gogamza/kobart-base-v2')
text = '검은 소가 누렁소보다 일을 더 잘합니다.'
print(kobart_tokenizer.tokenize(text))


['▁검은', '▁소', '가', '▁누', '렁', '소', '보다', '▁일을', '▁더', '▁잘', '합니다.']


## [코드 12-4] 토크나이저 호출 방식과 반환값

`tokenize()` 대신 객체 호출(`tokenizer(text)`) 형태로 직접 사용하면 토큰 고유
번호 리스트(`input_ids`)와 패딩 마스크(`attention_mask`)가 담긴 딕셔너리를
반환한다. `return_tensors='pt'`를 주면 파이토치 텐서로 받는다.


In [5]:
# 코드 12-4 - 토크나이저 호출 방식과 반환값
text = '검은 소가 누렁소보다 일을 더 잘합니다.'
list_result = kobart_tokenizer(text)                      # 리스트로 결과 반환
pt_result = kobart_tokenizer(text, return_tensors='pt')   # 파이토치 텐서로 반환
print('리스트 반환:')
print(list_result)
print()
print('파이토치 텐서 반환:')
print(pt_result)


리스트 반환:
{'input_ids': [19628, 14081, 8981, 14402, 10295, 11319, 14310, 15462, 14166, 14334, 20357], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

파이토치 텐서 반환:
{'input_ids': tensor([[19628, 14081,  8981, 14402, 10295, 11319, 14310, 15462, 14166, 14334,
         20357]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


## [코드 12-5] 대화틀 함수

지시어 튜닝 모델은 역할(role)과 내용(content)으로 이루어진 딕셔너리의 리스트를
입력으로 받는다. 역할에는 `system`(행동 방식), `user`(질문), `assistant`(과거
답변)가 있다.


In [9]:
# 코드 12-5 - 대화틀 함수
def chat_template(prompt):
    message = [
        {'role': 'system',
         'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'},
        {'role': 'user', 'content': prompt},
    ]
    return message


example_messages = chat_template('안녕? 오늘 날씨가 좋구나!')
print(example_messages)


[{'role': 'system', 'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'}, {'role': 'user', 'content': '안녕? 오늘 날씨가 좋구나!'}]


## [코드 12-6] 대화 메시지를 모델 입력 텐서로 변환

대화틀로 만든 메시지는 토크나이저의 `apply_chat_template()`을 거쳐 모델 입력
텐서(`input_ids`)로 인코딩된다. 역할 구분에 필요한 특수 토큰은 토크나이저가
모델에 맞춰 자동으로 끼워 넣는다.


In [10]:
# 코드 12-6 - 대화 메시지를 모델 입력 텐서로 변환
def generate_message(prompt, device):
    messages = chat_template(prompt)
    # apply_chat_template은 모델 학습 형식대로 입력을 재구성한다.
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=False,                 # input_ids 텐서만 반환
    ).to(device)
    return input_ids


example_input_ids = generate_message('안녕? 오늘 날씨가 좋구나!', device)
print(f'입력 텐서 shape: {example_input_ids.shape}')   # (1, S)
print(f'디코딩 결과:\n{tokenizer.decode(example_input_ids[0])}')


입력 텐서 shape: torch.Size([1, 61])
디코딩 결과:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 19 Jun 2026

당신은 한국어를 사용하는 친절한 AI 친구입니다.<|eot_id|><|start_header_id|>user<|end_header_id|>

안녕? 오늘 날씨가 좋구나!<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## [코드 12-7] LLM에 대화 메시지를 입력해 답변 생성

모델의 `generate()` 메서드에 입력 텐서를 넣으면 답변 토큰이 생성된다. LLaMA
계열은 종료 토큰이 두 종류(`<|end_of_text|>`, `<|eot_id|>`)라 둘 다 리스트
(`terminators`)로 묶어 `eos_token_id`에 넘긴다. 출력 텐서에는 입력 토큰까지
포함되므로 입력 길이(`input_ids.shape[-1]`) 이후만 디코딩한다.


In [11]:
# 코드 12-7 - LLM에 대화 메시지를 입력해 답변 생성
# 모델이 사용하는 종료 특수 토큰 두 종류를 모두 종료 토큰으로 지정
terminators = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]
prompt = '안녕? 오늘 날씨가 좋구나!'
max_new_tokens = 512
input_ids = generate_message(prompt, device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,  # 최대 생성 토큰 길이
        eos_token_id=terminators,
        do_sample=True,                 # 확률에 기반해 토큰 샘플링
        temperature=0.6,                # 생성 온도 (낮을수록 보수적)
        top_p=0.9,                      # 누적 확률 90% 안의 후보에서 선택
        pad_token_id=tokenizer.eos_token_id,
    )

# 출력 텐서에는 입력 토큰까지 포함되므로 입력 길이만큼 잘라 새 토큰만 디코딩
llm_generated = output_ids[0][input_ids.shape[-1]:]
result_text = tokenizer.decode(llm_generated, skip_special_tokens=True)
print(f'사용자 프롬프트 : {prompt}')
print(f'LLM의 답변 : {result_text}')


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


사용자 프롬프트 : 안녕? 오늘 날씨가 좋구나!
LLM의 답변 : 안녕하세요! 날씨가 좋네요? 오늘 어떤 계획이 있나요?
